In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# الإعدادات الافتراضية وخيارات الضبط للـ Transpiler


<details>
<summary><b>إصدارات الحزم</b></summary>

تمّ تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
ننصح باستخدام هذه الإصدارات أو أحدث منها.

```
qiskit[all]~=2.3.0
qiskit-ibm-runtime~=0.43.1
```
</details>

تحتاج الدوائر المجردة إلى transpilation لأن وحدات المعالجة الكمومية (QPUs) تمتلك مجموعة محدودة من البوابات الأساسية ولا تستطيع تنفيذ عمليات اعتباطية. وظيفة الـ Transpiler هي تحويل هذه الدوائر الاعتباطية لتتمكن من العمل على وحدة QPU محددة. يتم ذلك عبر ترجمة الدوائر إلى البوابات الأساسية المدعومة، وإدخال بوابات SWAP عند الحاجة، بحيث تتوافق اتصالية الدائرة مع اتصالية وحدة QPU.

كما هو موضح في [الـ Transpile باستخدام pass managers](transpile-with-pass-managers)، يمكنك إنشاء [pass manager](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager) باستخدام دالة [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager) وتمرير دائرة أو قائمة من الدوائر إلى طريقة [run](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.PassManager#run) الخاصة بها لإجراء الـ transpilation. يمكنك استدعاء `generate_preset_pass_manager` بتمرير مستوى التحسين والـ backend فقط، مع الاكتفاء بالقيم الافتراضية لبقية الخيارات، أو يمكنك تمرير وسائط إضافية لضبط عملية الـ transpilation بدقة أكبر.

## الاستخدام الأساسي بدون معاملات
في هذا المثال، نمرر دائرة ووحدة QPU المستهدفة إلى الـ Transpiler دون تحديد أي معاملات إضافية.

أنشئ دائرة واعرض النتيجة:

In [1]:
from qiskit import QuantumCircuit
from qiskit.circuit.library import grover_operator, DiagonalGate
from qiskit_ibm_runtime.fake_provider import FakeSherbrooke

# Create circuit to test transpiler on
oracle = DiagonalGate([1] * 7 + [-1])
qc = QuantumCircuit(3)
qc.h([0, 1, 2])
qc = qc.compose(grover_operator(oracle))

# Add measurements to the circuit
qc.measure_all()

# View the circuit
qc.draw(output="mpl")

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg" alt="Output of the previous code cell" />

![مخرجات خلية الكود السابقة](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/f7070db2-2b3e-4dcd-bbc7-cac7662867b3-0.svg)

اعمل على transpile للدائرة واعرض النتيجة:

In [2]:
from qiskit.transpiler import generate_preset_pass_manager

# Specify the QPU to target
backend = FakeSherbrooke()

# Transpile the circuit
pass_manager = generate_preset_pass_manager(
    optimization_level=1, backend=backend
)
transpiled_circ = pass_manager.run(qc)

# View the transpiled circuit
transpiled_circ.draw(output="mpl", idle_wires=False)

<Image src="../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg" alt="Output of the previous code cell" />

![مخرجات خلية الكود السابقة](../docs/images/guides/defaults-and-configuration-options/extracted-outputs/27ab746b-e3d7-49a7-b40b-d1e2d9ca6088-0.svg)

## جميع المعاملات المتاحة
فيما يلي جميع المعاملات المتاحة لدالة [`generate_preset_pass_manager`](https://docs.quantum.ibm.com/api/qiskit/qiskit.transpiler.generate_preset_pass_manager#qiskit.transpiler.generate_preset_pass_manager). هناك فئتان من الوسائط: تلك التي تصف هدف التجميع (compilation target)، وتلك التي تؤثر في طريقة عمل الـ Transpiler.

جميع المعاملات ما عدا `optimization_level` اختيارية. للاطلاع على التفاصيل الكاملة، راجع [توثيق Transpiler API](https://docs.quantum.ibm.com/api/qiskit/transpiler#transpiler-api).

- `optimization_level` (int) - مقدار التحسين الذي يُجرى على الدوائر. عدد صحيح في النطاق (0 - 3). المستويات الأعلى تُنتج دوائر أكثر تحسينًا، لكن على حساب وقت أطول للـ transpilation. راجع [ضبط مستوى التحسين للـ Transpiler](set-optimization) لمزيد من التفاصيل.

### المعاملات المستخدمة لوصف هدف التجميع:
تصف هذه الوسائط وحدة QPU المستهدفة لتنفيذ الدائرة، وتشمل معلومات مثل خريطة الاقتران (coupling map) لوحدة QPU (التي تصف اتصالية الـ Qubits)، والبوابات الأساسية التي تدعمها وحدة QPU، ومعدلات خطأ البوابات.

كثير من هذه المعاملات مشروحة بالتفصيل في [المعاملات الشائعة الاستخدام للـ transpilation](common-parameters).

<details>
  <summary>
    **معاملات وحدة QPU (`Backend`)**
  </summary>

**معاملات الـ Backend** - إذا حددت `backend`، فلن تحتاج إلى تحديد `target` أو أي خيارات backend أخرى. وبالمثل، إذا حددت `target`، فلن تحتاج إلى تحديد `backend` أو أي خيارات backend أخرى.
- `backend` (Backend) - إذا تم ضبط هذا، يقوم الـ Transpiler بتجميع دائرة الإدخال لهذا الجهاز. إذا تم ضبط أي خيار آخر يؤثر في هذه الإعدادات، مثل `coupling_map`، فإنه يتجاوز الإعدادات الواردة من `backend`.
- `target` (Target) - هدف الـ Transpiler للـ backend. عادةً ما يُحدد هذا كجزء من وسيطة الـ backend، لكن إذا قمت بإنشاء كائن Target يدويًا، يمكنك تحديده هنا. يتجاوز هذا الهدف المُحدد من `backend`.
- `backend_properties` (BackendProperties) - الخصائص التي تُعيدها وحدة QPU، بما فيها معلومات عن أخطاء البوابات وأخطاء القراءة وأوقات التماسك للـ Qubits وغيرها. اعثر على وحدة QPU تُوفر هذه المعلومات عبر تشغيل `backend.properties()`.
- `timing_constraints` (Dict[str, int] | None) - قيد اختياري على الأجهزة يتعلق بدقة توقيت التعليمات. تُوفر هذه المعلومات من إعداد وحدة QPU. إذا لم تكن لوحدة QPU أي قيود على توزيع وقت التعليمات، فإن `timing_constraints` تكون `None` ولا يُجرى أي تعديل. قد تُبلغ وحدة QPU عن مجموعة من القيود، وهي:
    - `granularity`: قيمة صحيحة تمثل الحد الأدنى لدقة بوابة النبضة (pulse gate) بوحدات dt. يجب أن تكون مدة بوابة النبضة المعرَّفة من قبل المستخدم من مضاعفات هذه القيمة.
    - `min_length`: قيمة صحيحة تمثل الحد الأدنى لطول بوابة النبضة بوحدات dt. يجب أن تكون بوابة النبضة المعرَّفة من قبل المستخدم أطول من هذا الحد.
    - `pulse_alignment`: قيمة صحيحة تمثل دقة توقيت وقت بدء تعليمات البوابة. يجب أن تبدأ تعليمات البوابة في وقت يمثل مضاعفًا لهذه القيمة.
    - `acquire_alignment`: قيمة صحيحة تمثل دقة توقيت وقت بدء تعليمات القياس. يجب أن تبدأ تعليمات القياس في وقت يمثل مضاعفًا لهذه القيمة.
</details>

<details>
  <summary>
    **معاملات التخطيط والطوبولوجيا**
  </summary>

- `basis_gates` (List[str] | None) - قائمة بأسماء البوابات الأساسية للتوسع إليها. مثلاً ['u1', 'u2', 'u3', 'cx']. إذا كانت `None`، لا يتم التوسع.
- `coupling_map` (CouplingMap | List[List[int]]) - خريطة اقتران موجَّهة (ربما مخصصة) تُستهدف في عملية التعيين. إذا كانت خريطة الاقتران متماثلة، يجب تحديد الاتجاهين. الصيغ المدعومة هي:
    - نسخة CouplingMap
    - قائمة (List) - يجب إعطاؤها كمصفوفة تجاور، حيث تُحدد كل إدخال جميع التفاعلات الثنائية الموجَّهة بين الـ Qubits التي تدعمها وحدة QPU. مثلاً: [[0, 1], [0, 3], [1, 2], [1, 5], [2, 5], [4, 1], [5, 3]]
- `inst_map` (List[InstructionScheduleMap] | None) - تعيين عمليات الدائرة إلى جداول النبضات. إذا كانت `None`، يُستخدم `instruction_schedule_map` الخاص بوحدة QPU.
</details>

### المعاملات المستخدمة للتأثير في طريقة عمل الـ Transpiler
تؤثر هذه المعاملات في مراحل transpilation محددة. بعضها قد يؤثر في مراحل متعددة، لكنه مدرج تحت مرحلة واحدة فقط للتبسيط. إذا حددت وسيطةً ما، مثل `initial_layout` للـ Qubits التي تريد استخدامها، فإن هذه القيمة تتجاوز جميع المسارات (passes) التي يمكنها تغييرها. بمعنى آخر، لن يغيّر الـ Transpiler أي شيء قمت بتحديده يدويًا. للاطلاع على تفاصيل المراحل المحددة، راجع [مراحل الـ Transpiler](transpiler-stages).

<details>
  <summary>
    **مرحلة التهيئة**
  </summary>

- `hls_config` (HLSConfig) - فئة إعداد اختيارية `HLSConfig` تُمرر مباشرةً إلى مسار التحويل `HighLevelSynthesis`. تتيح لك هذه الفئة تحديد قوائم خوارزميات التوليف ومعاملاتها لمختلف الكائنات عالية المستوى.
- `init_method` (str) - اسم الإضافة (plugin) المستخدمة لمرحلة التهيئة. بشكل افتراضي، لا تُستخدم أي إضافة خارجية. يمكنك رؤية قائمة الإضافات المثبتة عبر تشغيل `list_stage_plugins()` مع `init` كوسيطة لاسم المرحلة.
- `unitary_synthesis_method` (str) - اسم طريقة توليف الـ unitary المستخدمة. بشكل افتراضي، تُستخدم `default`. يمكنك رؤية قائمة الإضافات المثبتة عبر تشغيل `unitary_synthesis_plugin_names()`.
- `unitary_synthesis_plugin_config` (dict) - قاموس إعداد اختياري يُمرر مباشرةً إلى إضافة توليف الـ unitary. بشكل افتراضي، لا يكون لهذا الإعداد أي تأثير لأن طريقة توليف الـ unitary الافتراضية لا تقبل إعدادات مخصصة. يكون تطبيق إعداد مخصص ضروريًا فقط عند تحديد إضافة توليف unitary مع وسيطة `unitary_synthesis`. بما أن هذا خاص بكل إضافة، راجع توثيق الإضافة لمعرفة كيفية استخدام هذا الخيار.
</details>

<details>
  <summary>
    **مرحلة التخطيط (Layout)**
  </summary>

- `initial_layout` (Layout | Dict | List) - الموضع الأولي للـ Qubits الافتراضية على الـ Qubits الفيزيائية. إذا جعل هذا التخطيط الدائرة متوافقة مع قيود `coupling_map`، فسيُستخدم. لا يُضمن أن يكون التخطيط النهائي هو نفسه، إذ قد يُعيد الـ Transpiler ترتيب الـ Qubits عبر SWAP أو وسائل أخرى. للاطلاع على التفاصيل الكاملة، راجع [قسم التخطيط الأولي.](common-parameters#initial-layout)
- `layout_method` (str) - اسم مسار اختيار التخطيط (`default`، `dense`، `sabre`، و`trivial`). يمكن أن يكون هذا أيضًا اسم الإضافة الخارجية المستخدمة لمرحلة التخطيط. يمكنك رؤية قائمة الإضافات المثبتة عبر تشغيل `list_stage_plugins()` مع `layout` كوسيطة لـ `stage_name`. القيمة الافتراضية هي `sabre`.
</details>

<details>
  <summary>
    **مرحلة التوجيه (Routing)**
  </summary>

- `routing_method` (str) - اسم مسار التوجيه (`basic`، `lookahead`، `default`، `sabre`، أو `none`). يمكن أن يكون هذا أيضًا اسم الإضافة الخارجية المستخدمة لمرحلة التوجيه. يمكنك رؤية قائمة الإضافات المثبتة عبر تشغيل `list_stage_plugins()` مع `routing` كوسيطة لـ `stage_name`. القيمة الافتراضية هي `sabre`.
</details>

<details>
  <summary>
    **مرحلة الترجمة (Translation)**
  </summary>

- `translation_method` (str) - اسم مسار الترجمة (`default`، `synthesis`، `translator`، `ibm_backend`، `ibm_dynamic_circuits`، `ibm_fractional`). يمكن أن يكون هذا أيضًا اسم الإضافة الخارجية المستخدمة لمرحلة الترجمة. يمكنك رؤية قائمة الإضافات المثبتة عبر تشغيل `list_stage_plugins()` مع `translation` كوسيطة لـ `stage_name`. القيمة الافتراضية هي `translator`.
</details>

<details>
  <summary>
    **مرحلة التحسين (Optimization)**
  </summary>

- `approximation_degree` (float، في النطاق 0-1 | None) - مقياس هيوريستي يُستخدم لتقريب الدائرة (1.0 = بدون تقريب، 0.0 = تقريب أقصى). القيمة الافتراضية هي 1.0. تحديد `None` يضبط درجة التقريب على معدل الخطأ المُبلَّغ عنه. راجع [قسم درجة التقريب](common-parameters#approx-degree) لمزيد من التفاصيل.
- `optimization_method` (str) - اسم الإضافة المستخدمة لمرحلة التحسين. بشكل افتراضي، لا تُستخدم أي إضافة خارجية. يمكنك رؤية قائمة الإضافات المثبتة عبر تشغيل `list_stage_plugins()` مع `optimization` كوسيطة لـ `stage_name`.
</details>

<details>
  <summary>
    **الجدولة (Scheduling)**
  </summary>

- `scheduling_method` (str) - اسم مسار الجدولة. يمكن أن يكون هذا أيضًا اسم الإضافة الخارجية المستخدمة لمرحلة الجدولة. يمكنك رؤية قائمة الإضافات المثبتة عبر تشغيل `list_stage_plugins()` مع `scheduling` كوسيطة لـ `stage_name`.
  * 'as_soon_as_possible': جدولة التعليمات بشكل جشع (greedy)، في أبكر وقت ممكن على مورد Qubit (الاختصار: `asap`).
  * 'as_late_as_possible': جدولة التعليمات متأخرًا، أي الإبقاء على الـ Qubits في الحالة الأساسية (ground state) قدر الإمكان (الاختصار: `alap`). هذا هو الإعداد الافتراضي.
</details>

<details>
  <summary>
    **تنفيذ الـ Transpiler**
  </summary>

- `seed_transpiler` (int) - يضبط البذور العشوائية (random seeds) للأجزاء العشوائية في الـ Transpiler.
</details>

تُستخدم القيم الافتراضية التالية إذا لم تُحدد أيًا من المعاملات السابقة. راجع [صفحة مرجع API](../api/qiskit/transpiler_preset) للطريقة لمزيد من المعلومات:

In [3]:
generate_preset_pass_manager(
    optimization_level=1,
    backend=None,
    target=None,
    basis_gates=None,
    coupling_map=None,
    initial_layout=None,
    layout_method=None,
    routing_method=None,
    translation_method=None,
    scheduling_method=None,
    approximation_degree=1.0,
    seed_transpiler=None,
    unitary_synthesis_method="default",
    unitary_synthesis_plugin_config=None,
    hls_config=None,
    init_method=None,
    optimization_method=None,
)

## Next steps

<Admonition type="tip" title="Recommendations">
    - Learn how to [Set the optimization level](set-optimization).
    - Review more [Commonly used parameters](common-parameters).
    - Learn how to [Set the optimization level when using Qiskit Runtime.](./runtime-options-overview)
    - Visit the [Transpile with pass managers](transpile-with-pass-managers) topic.
    - For examples, see [Representing quantum computers.](./represent-quantum-computers)
    - Learn [how to transpile circuits](/docs/guides/circuit-transpilation-settings) as part of the Qiskit patterns workflow using Qiskit Runtime.
    - Review the [Transpile API documentation](/docs/api/qiskit/transpiler).
</Admonition>